# Temporal Reasoning Benchmark Creation

In [ ]:
from tqdm import tqdm
from src import SPARQL
import pandas as pd
import random
import re
import os
# Seed for random states
seed= 42
rng = random.Random(seed)


politicaltr_datapath= "data/tr_benchmarks/politicaltr"
politicaltr_raw_datapath= f"{politicaltr_datapath}/raw"

# For politicaltr
min_size= 70 # Minimum number of words to be considered a long text (only the text with this size will be taken for the politicaltr benchmark)
test_percentage = 0.75 # Define the percentage for train and test in politicaltr benchmark

# For tram
max_tram_instances= 5000 # Maximum number of instances per task to be taken from TRAM benchmark (to limit the size of the benchmark)
tram_datapath= f"data/tr_benchmarks/TRAM"
tram_raw_datapath= f"{tram_datapath}/raw"


## SPARQL Utils

In [ ]:
# Prefixes dict of all the KG
prefixes= { "dc": "http://purl.org/dc/elements/1.1/",
            "dcam": "http://purl.org/dc/dcam/",
            "eli": "http://data.europa.eu/eli/ontology#",
            "foaf": "http://xmlns.com/foaf/0.1/",
            "lkg": "http://lkg.lynx-project.eu/def/",
            "nif-core": "http://persistence.uni-leipzig.org/nlp2rdf/ontologies/nif-core#",
            "owl": "http://www.w3.org/2002/07/owl#",
            "podio": "http://w3id.org/podio#",
            "rdf": "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
            "schema": "http://schema.org/",
            "sioc": "http://rdfs.org/sioc/ns#",
            "skos": "http://www.w3.org/2004/02/skos/core#",
            "terms": "http://purl.org/dc/terms/",
            "vann": "http://purl.org/vocab/vann/",
            "xml": "http://www.w3.org/XML/1998/namespace",
            "xsd": "http://www.w3.org/2001/XMLSchema#",
            "rdfs": "http://www.w3.org/2000/01/rdf-schema#",
            "nif": "http://persistence.uni-leipzig.org/nlp2rdf/ontologies/nif-core#",
            "wd": "http://www.wikidata.org/entity/",
            "wdt": "http://www.wikidata.org/prop/direct/",
            "wikibase": "http://wikiba.se/ontology#",
            "bd": "http://www.bigdata.com/rdf#",
            "ps": "http://www.wikidata.org/prop/statement/",
            "p": "http://www.wikidata.org/prop/",
            "pq": "http://www.wikidata.org/prop/qualifier/",
            "ptr": "http://w3id.org/podio/resource/",
            "time": "http://www.w3.org/2006/time#"
            }

# Prefixes declaration during all the sparql queries
PREFIXES = """""".join(f"PREFIX {k}: <{v}>\n" for k, v in prefixes.items()) + """"""

# TR-KG endpoint declaration
GDB_REPO= "politicaltr"
GDB_PORT= 7300
GDB_URI= f"http://0.0.0.0:{GDB_PORT}/repositories/{GDB_REPO}"
GDB_ENDPOINT= SPARQL(endpoint=GDB_URI)


## Political Discourse Temporal Reasoning Benchmark Creation

### Cases Definition

First of all, we want to determine how many cases meet our criteria (100 minimum words) in each interval. In this case, the one with the fewest discourses contains 4. We obtained this information as follows:

In [ ]:
def get_period_ndiscourses_minwords(min_words=100):
    query= f"""
        SELECT DISTINCT (COUNT(?discourse) AS ?n) ?period ?periodLabel
        WHERE {{
            ?discourse a podio:Discourse ;
                time:inside ?period ;
                schema:wordCount ?words.
            ?period rdfs:label ?periodLabel .
            FILTER(?words >= {min_words}) .
            }}
            GROUP BY ?period ?periodLabel ORDER BY ASC(?n)
    """
    return GDB_ENDPOINT.run_query_pandas(f"{PREFIXES} {query}")

discourses_per_period= get_period_ndiscourses_minwords(min_size)
min_discourses_per_period= int(discourses_per_period.iloc[0]['n'])
print(f"There are {len(discourses_per_period)} periods.\nThe one with less number of discourses is {discourses_per_period.iloc[0]['periodLabel']}, with only {min_discourses_per_period}.")
discourses_per_period.head(n=len(discourses_per_period))

Once this was known, we decided to extract 4 random discourses from each period that meet the same word count requirement as before. Since SPARQL does not allow for reproducible random selection, we perform this step locally using a fixed seed.

In [ ]:
def get_discourses_by_period_minwords(min_words=100):
    query = f"""
        SELECT ?discourse ?period ?periodLabel ?content
        WHERE {{
            ?discourse a podio:Discourse ;
                       time:inside ?period ;
                       schema:wordCount ?words ;
                       podio:content ?content .
            ?period rdfs:label ?periodLabel .
            FILTER(?words >= {min_words})
        }}
    """
    return GDB_ENDPOINT.run_query_pandas(f"{PREFIXES} {query}")

# Take all discourses
discourses= get_discourses_by_period_minwords(min_words=min_size)

# Take the examples to test TR
# Split the discourses into train and test for each period
test_discourse_cases = []
train_discourse_cases = []

for period, group in discourses.groupby("period"):
    # Calculate the number of test cases for this period
    test_cases = int(len(group) * test_percentage)
    
    # Sample test cases
    test_sample = group.sample(n=test_cases, random_state=seed)
    test_discourse_cases.extend(test_sample["discourse"].tolist())
    
    # Remaining discourses are used for training
    remaining_group = group.drop(test_sample.index)
    train_discourse_cases.extend(remaining_group["discourse"].tolist())


In [ ]:
# Dump cases to csv to be used in the experiments
pd.DataFrame({"discourse": train_discourse_cases}).to_csv(f"{politicaltr_raw_datapath}/train_discourses.csv", index=False)
pd.DataFrame({"discourse": test_discourse_cases}).to_csv(f"{politicaltr_raw_datapath}/test_discourses.csv", index=False)

pd.DataFrame({"manifesto": list(set([re.sub(r'-\d+$', '', uri) for uri in train_discourse_cases]))}).to_csv(f"{politicaltr_raw_datapath}/train_manifestos.csv", index=False)
pd.DataFrame({"manifesto": list(set([re.sub(r'-\d+$', '', uri) for uri in test_discourse_cases]))}).to_csv(f"{politicaltr_raw_datapath}/test_manifestos.csv", index=False)

### Benchmark Creation

| **Task**           | **Prompt** |
|--------------------|------------|
| **Ordering**       | **Q:** Which of the following periods directly preceded the publication of the following political discourse: `$discourse$`?  <br> A) `$period_1$` B) `$period_2$` C) `$period_3$` <br><br> **Q:** Which of the following periods directly followed the publication of the following political discourse: `$discourse$`?  <br> A) `$period_1$` B) `$period_2$` C) `$period_3$` <br><br> **Q:** Which of the following political discourses directly preceded the period: `$period$`?  <br> A) `$discourse_1$` B) `$discourse_2$` C) `$discourse_3$` <br><br> **Q:** Which of the following political discourses directly followed the period: `$period$`?  <br> A) `$discourse_1$` B) `$discourse_2$` C) `$discourse_3$` |
| **Typical Time**   | **Q:** During which of the following periods was published the political discourse: `$discourse$`?  <br> A) `$period_1$` B) `$period_2$` C) `$period_3$` <br><br> **Q:** Which of the following political discourses was published during the period: `$period$`?  <br> A) `$discourse_1$` B) `$discourse_2$` C) `$discourse_3$` |
| **Temporal NLI**   | **Q:** Premise: `$discourse$` Hypothesis: Published during the period: `$period$`  <br> A) entail B) not entail <br><br> **Q:** Premise: `$discourse$` Hypothesis: Directly preceded the period: `$period$`  <br> A) entail B) not entail <br><br> **Q:** Premise: `$discourse$` Hypothesis: Directly followed the period: `$period$`  <br> A) entail B) not entail <br><br> **Q:** Premise: `$discourse_{truncated}$` Hypothesis: `$text$`  <br> A) entail B) not entail |
| **Storytelling**   | **Q:** Continue the following political discourse, which was published during `$period$`: `$discourse_{truncated}$`  <br> A) `$text_1$` B) `$text_2$` C) `$text_3$` <br><br> **Q:** Continue the following political discourse, which was published preceding `$period$`: `$discourse_{truncated}$`  <br> A) `$text_1$` B) `$text_2$` C) `$text_3$` <br><br> **Q:** Continue the following political discourse, which was published following `$period$`: `$discourse_{truncated}$`  <br> A) `$text_1$` B) `$text_2$` C) `$text_3$` |


In [ ]:
test_discourse_cases= pd.read_csv(f"{politicaltr_raw_datapath}/test_discourses.csv")["discourse"].tolist()
train_discourse_cases= pd.read_csv(f"{politicaltr_raw_datapath}/train_discourses.csv")["discourse"].tolist()

In [ ]:
def read_all_queries(directory="queries"):
    files_content = {}
    for root, _, files in os.walk(directory):
        for file in files:
            file_path = os.path.join(root, file)
            with open(file_path, "r", encoding="utf-8") as f:
                files_content[file_path] = f.read()
    return files_content

def get_discourse_case_info(discourse_id):
    query= f"""
        select ?discourse ?content ?periodLabel ?actorLabel ?description ?areaLabel where {{
            BIND(<{discourse_id}> AS ?discourse) .
            ?discourse  podio:content ?content ;
                        time:inside ?period ;
                        terms:creator ?creator;
                        terms:description ?description;
                        schema:areaServed ?areaServed.
            
            ?areaServed rdfs:label ?areaLabel .
            ?creator rdfs:label ?actorLabel .
            ?period rdfs:label ?periodLabel ;
                    terms:type "Campaign".
        }}
    """
    return GDB_ENDPOINT.run_query_pandas(f"{PREFIXES} {query}")
    
def generate_tr_questions(discourse_cases):
    """From a list of discourse IDs automatically generate the benchmark.

    Args:
        discourse_cases (list): list of ids in the KG

    Returns:
        pd.DataFrame: benchmark DataFrame 
    """
    query_templates= read_all_queries()
    queries= [] # list of queries pending to run
    for discourse_case in discourse_cases:
        for path, q in query_templates.items():
            path= path.replace(".sparql", "")
            task, query_id= path.split("/")[1:]
            q= q.replace("<KG_DISCOURSE_CASE_URI>", f"<{discourse_case}>")
            queries.append({"task": task, "query_id": query_id, "discourse": discourse_case, "query": q})
    queries_df= pd.DataFrame(queries)

    questions= []
    for idx, r in tqdm(queries_df.iterrows(), total=len(queries_df)):
        question= GDB_ENDPOINT.run_query_dict(r["query"])
        if question["question"]:
            question["task"]= r["task"]
            question["query_id"]= r["query_id"]
            question["discourse"]= r["discourse"]
            questions.append(question)

    return pd.DataFrame(questions)


In [ ]:
# Generate Test TR set 
test = generate_tr_questions(test_discourse_cases)
test_info = pd.DataFrame([get_discourse_case_info(discourse_case).iloc[0].to_dict() for discourse_case in test_discourse_cases])

# Generate Train set (for in context learning)
train = generate_tr_questions(train_discourse_cases)
train_info = pd.DataFrame([get_discourse_case_info(discourse_case).iloc[0].to_dict() for discourse_case in train_discourse_cases])

In [ ]:
# Save the benchmark and examples for ICL
test.to_csv(f"{politicaltr_raw_datapath}/test.csv", index=False)
test_info.to_csv(f"{politicaltr_raw_datapath}/test_info.csv", index=False)

train.to_csv(f"{politicaltr_raw_datapath}/train.csv", index=False)
train_info.to_csv(f"{politicaltr_raw_datapath}/train_info.csv", index=False)

### Experiments creation

In [ ]:
# Some Utils for the experiments:
index2option= {0:"A)", 1:"B)", 2:"C)"}
option2index= {"A)":0, "B)":1, "C)":2}

def build_few_shot_prompt(row, train_df, n_shots=3, cot=False):
    task = row["task"]
    
    if task == "Temporal NLI": 
        n_options= 2
    else:
        n_options= 3

    # Construir el prompt
    prompt_parts = []
    
    prompt_parts.append("# Task\nAnswer the question by selecting the SINGLE correct option from the choices.")
    
    instructions ="# Output Rules (strict)\n"
    # instructions +="""- One line only. No extra text, no “Answer:”, no explanations, no leading/trailing spaces.
    # - Do not show reasoning or chain-of-thought.
    # - After printing the label, STOP generating.
    # - These Output Rules override any other instruction or example.\n"""
    
    instructions +="""- After printing the label, STOP generating. \n- Do not repeat the questions. \n- No extra text.\n- These rules override anything else.\n"""
      
    #Think step by step silently. Do NOT write your reasoning.
    prompt_parts.append(instructions)
    
    # Filtrar ejemplos del mismo task (ajusta si quieres que coincida también discourse_case u otro campo)
    relevant_examples = train_df[train_df["task"] == task]

    ## Aditional filtering like cosine similarity can be added
    
    # Elegir aleatoriamente n_shots ejemplos
    if n_shots >0:
        sampled = relevant_examples.sample(n=min(n_shots, len(relevant_examples)), random_state=seed)

        for _, ex in sampled.iterrows():
            prompt_parts.append(f"# Example \n{ex['question']}\nAnswer:\n{ex['answer']}\n")
    
    # Añadir el test objetivo
    prompt_parts.append(f"\n{row['question']}")
    
    if n_options == 3:
        prompt_parts.append(f"# {'Think step by step to select ' if cot else ''}The only correct option from choices A) B) or C) (case-sensitive) is:")
    else:
        prompt_parts.append(f"# {'Think step by step to select ' if cot else ''}The only correct option from options A) or B) (case-sensitive) is:")

    return "\n".join(prompt_parts)

# Load test set
test= pd.read_csv(f"{politicaltr_raw_datapath}/test.csv")
test_info= pd.read_csv(f"{politicaltr_raw_datapath}/test_info.csv")
# Load train set for ICL
train= pd.read_csv(f"{politicaltr_raw_datapath}/train.csv")
train_info= pd.read_csv(f"{politicaltr_raw_datapath}/train_info.csv")

# Crear ejemplos one shot y few-shot
test["0s_question"] = test.apply(lambda row: build_few_shot_prompt(row, train, 0, False), axis=1)
test["1s_question"] = test.apply(lambda row: build_few_shot_prompt(row, train, 1, False), axis=1)
test["3s_question"] = test.apply(lambda row: build_few_shot_prompt(row, train, 3, False), axis=1)

test["gold_label"] = test["answer"].map(option2index)
test.to_csv(f"{politicaltr_datapath}/test_complete.csv", index=False)


## TRAM Benchmark Recreation

### Data loading and transformation

In [ ]:
test_questions= []
train_questions= []

tram_nli= pd.read_csv(f"{tram_raw_datapath}/nli_mcq.csv")
tram_nli_shots= pd.read_csv(f"{tram_raw_datapath}/nli_shots_mcq.csv")

tram_nli["task"] = "Temporal NLI"
tram_nli_shots["task"] = "Temporal NLI"

####

tram_ordering= pd.read_csv(f"{tram_raw_datapath}/ordering_mcq.csv")
tram_ordering_shots= pd.read_csv(f"{tram_raw_datapath}/ordering_shots_mcq.csv")

tram_ordering = tram_ordering.rename(columns={"Category": "Source"})
tram_ordering_shots = tram_ordering_shots.rename(columns={"Category": "Source"})

tram_ordering["task"] = "Ordering"
tram_ordering_shots["task"] = "Ordering"

###

tram_storytelling= pd.read_csv(f"{tram_raw_datapath}/storytelling_mcq.csv")
tram_storytelling_shots= pd.read_csv(f"{tram_raw_datapath}/storytelling_shots_mcq.csv")

tram_storytelling["task"] = "Storytelling"
tram_storytelling_shots["task"] = "Storytelling"

###

tram_typical_time= pd.read_csv(f"{tram_raw_datapath}/typical_time_mcq.csv")
tram_typical_time_shots= pd.read_csv(f"{tram_raw_datapath}/typical_time_shots_mcq.csv")

tram_typical_time = tram_typical_time.rename(columns={"Category": "Source"})
tram_typical_time_shots = tram_typical_time_shots.rename(columns={"Category": "Source"})

tram_typical_time["task"] = "Typical Time"
tram_typical_time_shots["task"] = "Typical Time"

### Benchmark creation

In [ ]:
# Some Utils for the experiments:
index2option= {0:"A)", 1:"B)", 2:"C)"}
option2index= {"A)":0, "B)":1, "C)":2}

def create_options_and_instructions(correct_answer, distractors):
    all_options = distractors + [correct_answer]
    rng.shuffle(all_options)

    if len(all_options) == 3:
        instructions= "Respond only with one of the options \"A)\", \"B)\", or \"C)\" on a single line."
        options= f"A) {all_options[0]} \nB) {all_options[1]} \nC) {all_options[2]}" 
    elif len(all_options) == 2:
        instructions= "Respond only with one of the options \"A)\" or \"B)\" on a single line."
        options= f"A) {all_options[0]} \nB) {all_options[1]}" 
    
    gold_index = all_options.index(correct_answer)
    gold_label= index2option[gold_index]

    return {"instructions":instructions, "options":options, "n_options":len(all_options), "gold_label":gold_label}


def build_few_shot_prompt(row, train_df, n_shots=3, cot=False):
    task = row["task"]
    # Construir el prompt
    prompt_parts = []
    
    prompt_parts.append("# Task\nAnswer the question by selecting the SINGLE correct option from the choices.")
    
    instructions ="# Output Rules (strict)\n"
    # instructions +="""- One line only. No extra text, no “Answer:”, no explanations, no leading/trailing spaces.
    # - Do not show reasoning or chain-of-thought.
    # - After printing the label, STOP generating.
    # - These Output Rules override any other instruction or example.\n"""
    
    instructions +="""- After printing the label, STOP generating. \n- Do not repeat the questions. \n- No extra text.\n- These rules override anything else.\n"""
      
    #Think step by step silently. Do NOT write your reasoning.
    prompt_parts.append(instructions)
    
    # Filtrar ejemplos del mismo task (ajusta si quieres que coincida también discourse_case u otro campo)
    relevant_examples = train_df[train_df["task"] == task]

    ## Aditional filtering like cosine similarity can be added
    
    # Elegir aleatoriamente n_shots ejemplos
    if n_shots >0:
        sampled = relevant_examples.sample(n=min(n_shots, len(relevant_examples)), random_state=seed)

        for _, ex in sampled.iterrows():
            prompt_parts.append(f"# Example \n{ex['question']}\nAnswer:\n{ex['answer']}\n")
    
    # Añadir el test objetivo
    prompt_parts.append(f"\n{row['question']}")
    
    if row['n_options'] == 3:
        prompt_parts.append(f"# {'Think step by step to select ' if cot else ''}The only correct option from choices A) B) or C) (case-sensitive) is:")
    else:
        prompt_parts.append(f"# {'Think step by step to select ' if cot else ''}The only correct option from options A) or B) (case-sensitive) is:")

    return "\n".join(prompt_parts)

In [ ]:
nli_template= "Question:\nPremise: {premise}  Hypothesis: {hypothesis} \nChoices:\n {options}"
test_id_mapper= {"A": "T", "B": "N", "C": "F"}

for row_index, row in tram_nli.iterrows():
    test_elements= create_options_and_instructions( row[f"Option {row['Answer']}"],
                                                    [opt for opt in [row["Option A"], row["Option B"], row["Option C"]] if opt!= row[f"Option {row['Answer']}"]]
                                                                                                                              )
    question= nli_template.format(premise=row["Premise"], hypothesis= row["Hypothesis"], options=test_elements['options'])
    
    correct_option= test_elements["gold_label"]
    
    test_questions.append({
                      "task": row["task"],
                      "category": row["Source"],
                      "test_id": test_id_mapper[row["Answer"]],
                      "question": question, 
                      "answer":correct_option,
                      "n_options": test_elements['n_options'],
                      "gold_label": option2index[correct_option]
                     })

## Shots
for row_index, row in tram_nli_shots.iterrows():
    test_elements= create_options_and_instructions( row[f"Option {row['Answer']}"],
                                                    [opt for opt in [row["Option A"], row["Option B"], row["Option C"]] if opt!= row[f"Option {row['Answer']}"]]
                                                                                                                              )
    question= nli_template.format(premise=row["Premise"], hypothesis= row["Hypothesis"], options=test_elements['options'])
    
    correct_option= test_elements["gold_label"]
    
    train_questions.append({
                      "task": row["task"],
                      "category": row["Source"],
                      "test_id": test_id_mapper[row["Answer"]],
                      "0s_question": question, 
                      "answer":correct_option,
                      "n_options": test_elements['n_options'],
                      "gold_label": option2index[correct_option]
                     })


In [ ]:
simple_question_template= "Question:\n{question} \nChoices:\n {options}"

for df in [tram_ordering, tram_storytelling, tram_typical_time]:
    for row_index, row in df.iterrows():
        if "Option C" in row:
            test_elements= create_options_and_instructions( row[f"Option {row['Answer']}"],
                                                            [opt for opt in [row["Option A"], row["Option B"], row["Option C"]] if opt!= row[f"Option {row['Answer']}"]]
                                                                                                                                      )
        else:
            test_elements= create_options_and_instructions( row[f"Option {row['Answer']}"],
                                                            [opt for opt in [row["Option A"], row["Option B"]] if opt!= row[f"Option {row['Answer']}"]]
                                                                                                                                      )
        question= simple_question_template.format(question=row["Question"], options=test_elements['options'])

        correct_option= test_elements["gold_label"]

        test_questions.append({
                          "task": row["task"],
                          "category": row["Source"],
                          "test_id": row["Source"],
                          "question": question, 
                          "answer":correct_option,
                          "n_options": test_elements['n_options'],
                          "gold_label": option2index[correct_option]
                         })


for df in [tram_ordering_shots, tram_storytelling_shots, tram_typical_time_shots]:
    ## Shots
    for row_index, row in df.iterrows():
        if "Option C" in row:
            test_elements= create_options_and_instructions( row[f"Option {row['Answer']}"],
                                                            [opt for opt in [row["Option A"], row["Option B"], row["Option C"]] if opt!= row[f"Option {row['Answer']}"]]
                                                                                                                                      )
        else:
            test_elements= create_options_and_instructions( row[f"Option {row['Answer']}"],
                                                            [opt for opt in [row["Option A"], row["Option B"]] if opt!= row[f"Option {row['Answer']}"]]
                                                                                                                                      )
        question= simple_question_template.format(question=row["Question"], options=test_elements['options'])

        correct_option= test_elements["gold_label"]

        train_questions.append({
                          "task": row["task"],
                          "category": row["Source"],
                          "test_id": row["Source"],
                          "question": question, 
                          "answer":correct_option,
                          "n_options": test_elements['n_options'],
                          "gold_label": option2index[correct_option]
                         })

### Experiments creation

In [ ]:
test= pd.DataFrame(test_questions)
train= pd.DataFrame(train_questions)

test= (
    test.groupby("task", group_keys=False)
      .apply(lambda x: x.sample(n=max_tram_instances, random_state=seed))
      .reset_index(drop=True)
)

# Crear ejemplos one shot y few-shot
test["0s_question"] = test.apply(lambda row: build_few_shot_prompt(row, train, 0, False), axis=1)
test["1s_question"] = test.apply(lambda row: build_few_shot_prompt(row, train, 1, False), axis=1)
test["3s_question"] = test.apply(lambda row: build_few_shot_prompt(row, train, 3, False), axis=1)

test["gold_label"] = test["answer"].map(option2index)

test.to_csv(f"{tram_datapath}/tram_short.csv", index=False)